# 1단계 PLATOON1 — 무리 도착 세계에서 정책 반복 1라운드: 자료 모으기, 조각 0 / 3

PLATOON0에서 동쪽 차가 이웃 신호(204822)에 맞춰 무리로 오는 세계에서는 이웃 시간표를 쓰는 계획기(TM4)가 M4보다 3.82초
짧았습니다. 이번에는 그 정보를 보는 **강화학습(정책 반복)** 학생이 TM4보다 더 짧아지는지 봅니다. 기준은 실행 전에 고정했습니다
(`results/explore/platoon1_20261101/사전등록.md`).
- TM4가 학습 날짜의 무리 도착 상황 64판을 운전합니다. 고를 수 있는 순간의 25%에서 상태를 저장하고, **허용된 녹색을 모두**
  무리 규칙으로 만든 표본 미래 8개로 300초씩 굴려 봅니다(뒤는 TM4가 이어 운전).
- 다음 노트북 `PLATOON1_learn`에서 신경망이 '어느 녹색이 더 나은가'를 배우고 같은 심판으로 잽니다.

**이 파일 하나 = 런타임 하나입니다.** 조각 0 = `PLATOON1_data.ipynb`, 조각 1 = `PLATOON1_data_shard1.ipynb`,
조각 2 = `PLATOON1_data_shard2.ipynb`. 세 파일을 하나씩 열어(런타임 유형 **CPU + 높은 RAM**) 각각 **런타임 → 모두 실행**을 누르세요.
첫 칸의 `SHARD`는 이미 0로 정해져 있으니 바꾸지 마세요. 같은 조각을 두 런타임에서 동시에 돌리지 마세요.

순서: 묶음 검증 → 패키지 → 시험 → 실행 전 점검(진짜 SUMO, 약 20~30분) → 자료 모으기(백그라운드, 약 11~16시간) → 진행 확인.
끊기면 다시 **모두 실행**하면 끝난 판은 건너뜁니다. 마지막 칸에 **64/64**가 나오면 `PLATOON1_learn`을 실행하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_PLATOON1.zip"
PLATOON0 = DRIVE / "explore_platoon0"          # PLATOON0 결과(무리 도착 세계 M4·TM4·OM4·P1M4, 읽기만)
OUT = DRIVE / "explore_platoon1"
WORKSPACE = Path("/content/FutureTraffic_PLATOON1")
MIN_CPUS = 8
for path in (ARCHIVE, PLATOON0 / "experiment.json", PLATOON0 / "runs" / "P" / "TM4"):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
SHARD = 0          # 이 파일 = 조각 0(바꾸지 마세요)
SHARDS = 3         # 모든 조각 파일에서 같은 값
assert 0 <= SHARD < SHARDS
print("CPU 개수:", CPUS, "| 조각:", SHARD, "/", SHARDS, "| 결과 폴더:", OUT / "data")
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요. "
                          "(롤아웃을 CPU 수만큼 동시에 돌립니다. CPU 2개면 4배 오래 걸려 24시간 한도에 걸립니다.)")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "PLATOON1_V1", "PLATOON1 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("platoon1_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON1 설정 {package_manifest['platoon1_config_sha256'][:12]} | 사전 등록 {package_manifest['prereg_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

for name in ("test_explore_platoon1.py", "test_explore_platoon0.py", "test_arena3.py"):
    colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", name], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 실행 전 점검(약 20~30분): 무리 도착 학습 상황 하나에서
#    - TM4를 그냥 돌린 것과, 상태를 저장·롤아웃하며 돌린 것이 소수점까지 같은가,
#    - 학습 전 학생(신경망 출력 0)이 TM4와 같은가, 평가 상황에서 PLATOON0의 TM4와 같은가,
#    - 롤아웃 복원·TM4 기록 넘기기가 맞는가, 동쪽 표본 미래가 204822 녹색에 놓이는가, 판당 시간이 얼마인가.
SMOKE = OUT / "smoke" / f"shard{SHARD}"
previous = colab.read_json(SMOKE / "smoke.json")
if previous and previous.get("passed"):
    print("이전 점검이 통과해 건너뜁니다. 판당 약", round(previous["estimate_cpu_hours_per_episode"], 2), "CPU-시간")
else:
    colab.run_live(["src/explore_platoon1.py", "smoke", "--out", str(SMOKE), "--work", "/content/pt1_smoke",
                    "--workers", str(CPUS), "--platoon0", str(PLATOON0)], WORKSPACE, check=False)
s = colab.read_json(SMOKE / "smoke.json")
assert s and s.get("passed"), f"실행 전 점검 실패 — 이 칸의 출력을 저에게 보내 주세요: {s and s.get('checks')}"
print("점검 통과:", s["checks"])
print("판당 약", round(s["estimate_cpu_hours_per_episode"], 2), "CPU-시간, 64판 합 약", round(s["estimate_cpu_hours_all"]), "CPU-시간")

In [ ]:
# 2) 자료 모으기: 이 런타임 몫(64판 중 SHARD번째부터 3칸씩). 백그라운드로 돌아 셀을 멈춰도 계속 돕니다.
#    런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하세요. 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
DATA = OUT / "data"
LOGS = OUT / "logs"
NAME = f"collect_shard{SHARD}"


def describe(name):
    s = colab.read_json(DATA / "status" / f"{name}.json")
    if not s:
        return None
    return (f"끝난 판 {s['done']}/{s['total']}, 롤아웃 중 {len(s['inflight'])}판, 실패 {len(s['failed'])}, "
            f"복원 불일치 {s['restore_mismatched']}/{s['restore_checked']}")


pids = colab.launch({NAME: ["src/explore_platoon1.py", "collect", "--out", str(DATA), "--work", "/content/pt1",
                            "--shard", str(SHARD), "--shards", str(SHARDS), "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 600, describe)
assert code == 0, "자료 모으기에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."

In [ ]:
# 3) 모든 런타임의 진행(언제든 다시 실행해 볼 수 있습니다).
for f in sorted((OUT / "data" / "status").glob("collect_shard*.json")):
    s = colab.read_json(f) or {}
    print(f.stem, f"{s.get('done')}/{s.get('total')}", "| 실패", len(s.get("failed", [])),
          "| 복원 불일치", f"{s.get('restore_mismatched')}/{s.get('restore_checked')}")
done = len(list((OUT / "data" / "runs" / "PT1_DATA").glob("*/cache.json")))
print(f"끝난 판 {done}/64")
if done == 64:
    print("64판이 모두 끝났습니다. PLATOON1_learn 노트북을 실행하세요.")